# Data cleaning in SQL

General notes:

- We'll be using [ipython-sql](https://github.com/catherinedevlin/ipython-sql) to work with SQL directly within a notebook. To execute SQL commands, prefix with `%sql`. For multiple lines, use `%%sql` and then a newline.
- To use the [`psql` command line tool](https://www.postgresql.org/docs/current/app-psql.html), [launch a terminal](https://jupyterlab.readthedocs.io/en/stable/user/terminal.html) and run `psql postgresql://localhost/postgres`.
- If you get a database connection error, try [launching a terminal](https://jupyterlab.readthedocs.io/en/stable/user/terminal.html) and re-creating with `./start`.

## Environment setup

***You can probably ignore this section.***

In [ ]:
%load_ext sql

In [ ]:
%sql postgresql://localhost/postgres

Start from a clean slate:

In [ ]:
%sql DROP TABLE IF EXISTS requests;

### Load data

We're taking data from the CSV and putting it in an in-memory [SQLite](https://sqlite.org/index.html) database via [pandas](https://pandas.pydata.org/).

In [ ]:
import pandas as pd

# use a subset of columns, for simplicity
columns = [
    "Unique Key",
    "Created Date",
    "Closed Date",
    "Agency",
    "Latitude",
    "Longitude"
]
requests = pd.read_csv("data/311_jan_2022.csv", index_col="Unique Key", usecols=columns)
# simplify column names
requests.index = requests.index.rename("unique_key")
requests.columns = requests.columns.str.replace(" ", "_", regex=False).str.replace(r"\W", "", regex=True).str.lower()

In [ ]:
%sql --persist requests

#### Ensure records were loaded

In [ ]:
%sql SELECT * FROM requests LIMIT 3;

## [Display the schema](https://www.postgresql.org/docs/current/app-psql.html#APP-PSQL-META-COMMANDS)

In [ ]:
%sql \d requests

## Exploration

### Counts per unique value

Equivalent to [pandas' `value_counts()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html).

In [ ]:
%sql SELECT agency, COUNT(agency) FROM requests GROUP BY agency;

## Add constraints

[SQL constraints](https://www.postgresql.org/docs/current/ddl-constraints.html) allow you to enforce rules at the database level. This means bad data won't be let in. This then means an error will be throwing whenever bad data is coming in, so that needs to be handled by your applications.

Notes:

- `CHECK`s are somewhat limited in expressiveness
- The rules are one-size-fits-all
- You need to have write privileges on the database to implement them

### Primary key

In [ ]:
%sql ALTER TABLE requests ADD PRIMARY KEY (unique_key);

### Convert dates from strings to datetime

In [ ]:
%sql SELECT created_date FROM requests LIMIT 3;

In [ ]:
%%sql
ALTER TABLE requests
    ALTER COLUMN created_date SET NOT NULL,
    ALTER COLUMN created_date TYPE TIMESTAMP USING to_timestamp(created_date, 'MM-DD-YYYY HH:MI:SS AM');

Confirm they look the same:

In [ ]:
%sql SELECT created_date FROM requests LIMIT 3;

#### Your turn: Convert `closed_date` to a timestamp

In [ ]:
# YOUR CODE HERE

### Ranges

In [ ]:
%sql ALTER TABLE requests ADD CHECK (latitude > 40 AND latitude < 41);

## Clean columns

In [ ]:
%sql SELECT MIN(latitude), MAX(latitude) FROM requests;

This is an example where that close-to-zero number should probably be `null` instead. In other columns, values like `N/A` may have special meaning, or not. In each case, you have to decide what you want to do with them: exclude them, give default or derived values, etc. Here, we'll set them to `null`.

In [ ]:
%sql UPDATE requests SET latitude = null WHERE ABS(latitude) < 1;

### Your turn: Clean out the `longitude` column

In [ ]:
# YOUR CODE HERE

### Your turn: Try adding `CHECK` from above again

In [ ]:
# YOUR CODE HERE